# Partie C — Le produit gagnant

Source des données : Inside Airbnb (licence CC BY 4.0), snapshot : 2026-09-17.
Ville : Lyon.
Responsable : Membre 3. En appui : Membre 1 (C1, C4 selon Gantt), Membre 2 (C2, C3, C4).

> Tant que la table nettoyée de A6 n'existe pas (prévue le 19/11), ce notebook travaille sur les données brutes avec un nettoyage minimal. Ces résultats servent uniquement à **choisir les équipements et le périmètre**. Les chiffres définitifs viennent de la table nettoyée.

In [ ]:
import sys
sys.path.append("..")
from src.config import *
import re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.equipements import MOTIFS, parse_amenities, ajouter_indicatrices
from src.primes import (perimetre_logements_entiers, ajouter_classe_capacite, classes_valides,
                        tableau_primes)
np.random.seed(SEED)
print("Ville :", VILLE)
print("Dossier données :", DOSSIER, "| existe :", DOSSIER.exists())

## C0 — Chargement et périmètre

**Périmètre retenu : logements entiers, actifs, avec un prix publié.**
*Justification :* mélanger chambres et logements entiers fausse toutes les comparaisons (la taille et le standing diffèrent). Un logement entier est aussi ce que vise un investisseur. Un prix publié est nécessaire pour calculer les primes de prix, et une annonce active garantit que le revenu estimé est observable.
*Définition provisoire de « active » :* occupation estimée sur 12 mois strictement positive. À remplacer par la variable de B1 quand elle existera.

In [ ]:
COLS = ["id", "room_type", "price", "accommodates", "bedrooms", "amenities",
        "neighbourhood_cleansed", "estimated_occupancy_l365d", "estimated_revenue_l365d",
        "review_scores_rating", "host_is_superhost", "minimum_nights",
        "calculated_host_listings_count"]

FICHIER_PROPRE = CLEAN / "listings_propre.parquet"   # nom convenu avec l'équipe pour A6


def prix_en_nombre(v):
    """Conversion provisoire de price. Formats vus dans les données brutes de Lyon :
    '$1,250.00', '142,50 $' (virgule décimale), '136 dollars', 'CAD 184', '118.00', '-$90.00'.
    Le signe est ignoré (erreur de signe probable) : A5 tranchera pour la table nettoyée."""
    if pd.isna(v):
        return np.nan
    s = re.sub(r"[^\d.,]", "", str(v))
    if not s:
        return np.nan
    if "." not in s and re.search(r",\d{1,2}$", s):   # virgule décimale
        s = s.replace(",", ".")
    else:                                              # virgule = séparateur de milliers
        s = s.replace(",", "")
    return float(s)


if FICHIER_PROPRE.exists():
    listings = pd.read_parquet(FICHIER_PROPRE)
    SOURCE = "table nettoyée (A6)"
else:
    listings = pd.read_csv(DOSSIER / "listings.csv.gz", usecols=COLS)
    # Nettoyage minimal et provisoire : A3 et A5 feront le vrai traitement
    listings = listings.drop_duplicates(subset="id", keep="first")
    listings["price"] = listings["price"].map(prix_en_nombre)
    listings["neighbourhood_cleansed"] = listings["neighbourhood_cleansed"].str.strip().str.lower()
    listings = listings[listings["accommodates"].between(1, 16)]
    SOURCE = "données brutes (exploration provisoire)"

if not pd.api.types.is_bool_dtype(listings["host_is_superhost"]):
    # écritures vues : t/f, true/false, 1/0, yes/no, vrai/faux ; A3 fera le traitement complet
    listings["host_is_superhost"] = (
        listings["host_is_superhost"].astype(str).str.strip().str.lower()
        .map({"t": True, "true": True, "1": True, "yes": True, "vrai": True,
              "f": False, "false": False, "0": False, "no": False, "faux": False}))

print(SOURCE, "|", listings.shape)

# Entonnoir du périmètre
listings["active_provisoire"] = listings["estimated_occupancy_l365d"].fillna(0) > 0
entiers = listings["room_type"].astype(str).str.lower().str.contains("entire")
avec_prix = listings["price"].notna() & (listings["price"] > 0)
print("Annonces au total        :", len(listings))
print("... logements entiers    :", int(entiers.sum()))
print("... avec un prix publié  :", int((entiers & avec_prix).sum()))
perimetre = perimetre_logements_entiers(listings, "active_provisoire")
print("... actives (périmètre)  :", len(perimetre))

## C1 — La prime des équipements

### C1.1 — Choix des équipements
On part de 12 candidats plausibles pour un investisseur (voir `src/equipements.py`). On retient ceux qui sont **ni trop rares, ni quasi universels** (présence entre 5 % et 95 %), car un équipement que presque tout le monde a (ou n'a pas) ne permet aucune comparaison, et qui ont assez d'annonces pour une comparaison à capacité égale (au moins 2 classes de capacité avec 10 annonces équipées et 10 non équipées).

In [ ]:
perimetre = ajouter_indicatrices(perimetre)
perimetre = ajouter_classe_capacite(perimetre)

selection = pd.DataFrame({
    "part_equipees_%": (perimetre[list(MOTIFS)].mean() * 100).round(1),
    "classes_comparables": [classes_valides(perimetre, eq) for eq in MOTIFS],
})
selection["retenu"] = (selection["part_equipees_%"].between(5, 95)
                       & (selection["classes_comparables"] >= 2))
display(selection.sort_values("part_equipees_%", ascending=False))

EQUIPEMENTS_RETENUS = selection.index[selection["retenu"]].tolist()
print(len(EQUIPEMENTS_RETENUS), "équipements retenus :", EQUIPEMENTS_RETENUS)
if not 8 <= len(EQUIPEMENTS_RETENUS) <= 12:
    print("Attention : l'énoncé demande 8 à 12 équipements. Ajuster les seuils ou les motifs.")

*Liste finale validée en équipe (à compléter) :*

### C1.2 à C1.5 — Naïve, annonces comparables, régression
Trois méthodes sur les mêmes équipements :
1. **naïve** : écart des médianes équipées / non équipées ;
2. **annonces comparables** : même comparaison à l'intérieur de chaque classe de capacité (1-2, 3-4, 5-6, 7+), moyenne pondérée par la taille des groupes (annexe E) ;
3. **régression** : log(prix) sur tous les équipements, la capacité, les chambres et le quartier ; prime = exp(coefficient) − 1.

In [ ]:
tab = tableau_primes(perimetre, EQUIPEMENTS_RETENUS)
tab = tab.sort_values("comparable_revenu", ascending=False)
display((tab * 100).round(1).rename(columns=lambda c: c + " (%)"))

In [ ]:
# Les trois primes de prix côte à côte : voir l'écart entre méthodes (C1.6)
graph = tab[["naive_prix", "comparable_prix", "regression_prix"]].mul(100)
graph.columns = ["Naïve", "À capacité égale", "Régression"]
ax = graph.plot.barh(figsize=(9, 6))
ax.axvline(0, color="grey", linewidth=0.8)
ax.set_xlabel("Prime de prix (%)")
ax.set_ylabel("Équipement")
ax.set_title("La prime naïve surestime l'effet des équipements")
plt.tight_layout()
plt.show()

### C1.6 — Lecture
*À rédiger après exécution sur la table nettoyée :*
- pourquoi l'écart entre méthodes (les grands biens cumulent les équipements) ;
- équipements recommandés et niveau de confiance (un résultat qui change de signe d'une méthode à l'autre ne fonde aucune recommandation) ;
- limites : décoration, photos, réactivité de l'hôte non observées ; association et non causalité.

## C2 — L'arbitrage prix / occupation

Déciles du **prix par personne** (et non du prix brut, sinon les déciles les plus chers regroupent les grands biens : on mesurerait la taille, pas le niveau de prix).

In [ ]:
c2 = perimetre[perimetre["accommodates"] > 0].copy()
c2["prix_par_personne"] = c2["price"] / c2["accommodates"]
c2["decile_prix"] = pd.qcut(c2["prix_par_personne"], 10, labels=False, duplicates="drop") + 1

deciles = c2.groupby("decile_prix").agg(
    annonces=("id", "count"),
    prix_pers_median=("prix_par_personne", "median"),
    occupation_mediane=("estimated_occupancy_l365d", "median"),
    revenu_median=("estimated_revenue_l365d", "median"),
).round(1)
display(deciles)

meilleur = deciles["revenu_median"].idxmax()
print(f"Décile de prix au revenu médian le plus élevé : {meilleur} "
      f"(prix médian par personne : {deciles.loc[meilleur, 'prix_pers_median']} par nuit)")

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(deciles.index, deciles["occupation_mediane"], marker="o", color="tab:blue")
ax1.set_xlabel("Décile de prix par personne (1 = le moins cher)")
ax1.set_ylabel("Occupation médiane estimée (nuits sur 12 mois)", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(deciles.index, deciles["revenu_median"], marker="s", color="tab:orange")
ax2.set_ylabel("Revenu médian estimé (par an, devise locale)", color="tab:orange")
ax1.set_title("L'occupation baisse quand le prix monte : où le revenu est-il maximal ?")
plt.tight_layout()
plt.show()

# Fiabilité : plafond de l'occupation estimée
plafond = perimetre["estimated_occupancy_l365d"].max()
part_plafond = (perimetre["estimated_occupancy_l365d"] == plafond).mean()
print(f"Occupation maximale observée : {plafond} nuits ; annonces au plafond : {part_plafond:.1%}")

*Lecture (à rédiger) :* zone de prix qui maximise le revenu, et ce que change le plafond d'occupation (au-delà, on ne distingue plus les annonces entre elles : le revenu des annonces les mieux occupées est sous-estimé).

## C3 — Le profil des annonces les plus rentables

In [ ]:
c3 = perimetre.copy()
c3["nb_equipements"] = c3["amenities"].map(lambda v: len(parse_amenities(v)))
seuil = c3["estimated_revenue_l365d"].quantile(0.90)
c3["groupe"] = np.where(c3["estimated_revenue_l365d"] >= seuil, "Top 10 %", "Reste")

numeriques = ["accommodates", "bedrooms", "price", "nb_equipements", "review_scores_rating",
              "minimum_nights", "calculated_host_listings_count", "estimated_occupancy_l365d"]
comparaison = c3.groupby("groupe")[numeriques].median().T
comparaison.loc["host_is_superhost (%)"] = (
    c3.groupby("groupe")["host_is_superhost"].mean().mul(100).round(1))
display(comparaison.round(2))

top_quartiers = (c3[c3["groupe"] == "Top 10 %"]["neighbourhood_cleansed"]
                 .value_counts(normalize=True).head(5).mul(100).round(1))
print("Quartiers les plus représentés dans le top 10 % (%) :")
display(top_quartiers)

*Les trois ingrédients du succès (à rédiger).*
*Biais du survivant :* le fichier ne contient que les annonces encore présentes ; celles qui ont échoué ont disparu. Les profils « rentables » observés sont donc ceux des survivants : le revenu réel attendu d'un nouvel entrant est plus bas.

## C4 — Du revenu à la rentabilité

Les données ne contiennent aucun coût. Les hypothèses ci-dessous sont **provisoires et à sourcer** (tâche « hypothèses de coûts », 17 au 19/11). Chaque paramètre doit recevoir une source avant la note de décision.

In [ ]:
# HYPOTHÈSES PROVISOIRES : À SOURCER avant utilisation dans la note de décision
HYP = {
    "frais_plateforme_pct": 0.03,         # À SOURCER : commission hôte Airbnb
    "menage_par_sejour": 40.0,            # À SOURCER : coût d'un ménage à Lyon (€)
    "duree_moyenne_sejour_nuits": 3.0,    # À SOURCER ou à estimer depuis les avis
    "consommables_par_nuit": 3.0,         # À SOURCER (€)
    "charges_fixes_annuelles": 1800.0,    # À SOURCER : énergie, internet, entretien (€)
    "assurance_annuelle": 250.0,          # À SOURCER (€)
}

# Profil étudié : provisoirement tout le périmètre ; remplacer par le profil recommandé
profil = perimetre

lignes = []
for nom, q in [("bas", 0.25), ("médian", 0.50), ("haut", 0.75)]:
    revenu = profil["estimated_revenue_l365d"].quantile(q)
    nuits = profil["estimated_occupancy_l365d"].quantile(q)
    sejours = nuits / HYP["duree_moyenne_sejour_nuits"]
    net = (revenu * (1 - HYP["frais_plateforme_pct"])
           - sejours * HYP["menage_par_sejour"]
           - nuits * HYP["consommables_par_nuit"]
           - HYP["charges_fixes_annuelles"] - HYP["assurance_annuelle"])
    lignes.append({"scénario": nom, "revenu_brut": round(revenu), "nuits": round(nuits),
                   "revenu_net": round(net)})
display(pd.DataFrame(lignes).set_index("scénario"))

*Niveau avancé (à faire) :* comparer ce revenu net au loyer longue durée d'un bien équivalent dans le même quartier (source externe à citer) : c'est le coût d'opportunité.
*Impôts et taxes :* à documenter (régime fiscal du meublé de tourisme, taxe de séjour) avec sources officielles.